# Team reference notebook

This notebook is included as part of the original group submission. It is not attributed to Ali Abdoli Masinan. See [contributions](../../docs/CONTRIBUTIONS.md). Saved outputs were cleared. Local paths and missing helper dependencies remain documented in the [execution notes](../../docs/REPRODUCIBILITY.md).


In [ ]:
import hbsir
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
hbsir.setup(years="1392-1402",method="download_cleaned",download_source="mirror")

In [ ]:
import yaml

def extract_table_names(yaml_file_path):
    with open(yaml_file_path, 'r', encoding='utf-8') as file:
        schema = yaml.safe_load(file)

    table_names = set()

    # Extract top-level tables (direct keys)
    for key in schema.keys():
        if isinstance(schema[key], dict) and 'instructions' in schema[key]:
            table_names.add(key)

    # Extract tables from table_list sections
    for section in schema.values():
        if isinstance(section, dict) and 'table_list' in section:
            table_list = section['table_list']
            if isinstance(table_list, dict):
                for tables in table_list.values():
                    if isinstance(tables, list):
                        table_names.update(tables)
                    elif isinstance(tables, str):
                        table_names.add(tables)
            elif isinstance(table_list, list):
                table_names.update(table_list)

    # Remove None if present and sort alphabetically
    table_names.discard(None)
    return sorted(table_names)

# Usage example
yaml_file = r'./schema.yaml'
tables = extract_table_names(yaml_file)
print(tables)

In [ ]:
import os

years = range(1392, 1403)
tables = extract_table_names(yaml_file)

for year in years:
    for t in tables:
        try:
            print("Loading year", year)
            table = hbsir.load_table(t, years=year)
            directory_path = f'./csvs data/{t}/{year}'
            os.makedirs(directory_path, exist_ok=True)
            output = os.path.join(directory_path, f'{t}_{year}.csv')
            table.to_csv(output, index=False)
            print("Table", t, "done!")
        except Exception as e:
            print(f"Error processing year {year}: {e}")
            continue

In [ ]:
def csvfind(tableName , year ):
    return  pd.read_csv('./csvs data/' + tableName +"/"+str(year)+"/"+tableName+"_"+str(year)+".csv")

In [ ]:
expenditures_95 = csvfind(tableName="Expenditures", year=1395)
members_properties_95 = csvfind(tableName="members_properties", year=1395)

In [ ]:

expenditures_95 = hbsir.add_attribute(table=expenditures_95, name='Province',year_col="Year" ,id_col="ID")
expenditures_95 = hbsir.add_attribute(table=expenditures_95, name="Urban_Rural",year_col="Year" ,id_col="ID")
expenditures_95 = hbsir.add_attribute(table=expenditures_95, name="County",year_col="Year" ,id_col="ID")


In [ ]:
expenditures_95 \
    [expenditures_95["Table_Name"] == ("hotel")] \
    [expenditures_95["Gross_Expenditure"] != expenditures_95["Net_Expenditure"]] \
.head()

In [ ]:
from utils import *
expenditures_95_merged = expenditures_95 \
    [expenditures_95["Table_Name"].isin(("hotel", "entertainment"))] \
    .groupby(["ID"]) \
    ["Net_Expenditure"] \
    .sum()
merged = pd.merge(get_total_income(1395), expenditures_95_merged, on="ID")
merged.head()
head_edu = members_properties_95[members_properties_95["Relationship"] == "Head"][["ID", "Education_Level"]]
merged2 = pd.merge(merged, head_edu, on=["ID"])
merged2.head()
    #[expenditures_95["Gross_Expenditure"] != expenditures_95["Net_Expenditure"]] \
#.head()

In [ ]:
entertainment_exp_by_edu = merged2.groupby("Education_Level")[["Net_Expenditure", "Income"]].mean().sort_values(by="Income", ascending=False)

import matplotlib.pyplot as plt

entertainment_exp_by_edu.plot(kind="barh", figsize=(8, 6))
plt.xlabel("entertainment expenditures")
plt.ylabel("education")
plt.tight_layout()
plt.show()

In [ ]:
from utils import *
expenditures_95_merged = expenditures_95 \
    [expenditures_95["Table_Name"].isin(("education",))] \
    .groupby(["ID"]) \
.head()
#    ["Net_Expenditure"] \
#    .sum()
merged = pd.merge(get_total_income(1395), expenditures_95_merged, on="ID")
merged.head()
head_edu = members_properties_95[members_properties_95["Relationship"] == "Head"][["ID", "Education_Level"]]
merged2 = pd.merge(merged, head_edu, on=["ID"])
merged2.head()
    #[expenditures_95["Gross_Expenditure"] != expenditures_95["Net_Expenditure"]] \
#.head()

In [ ]:
import statsmodels.api as sm
from itertools import combinations


def regression_with_interactions(df, independent_vars, target_var):
    # انتخاب داده‌ها
    X = df[independent_vars]
    y = df[target_var]

    # تبدیل متغیرهای رشته‌ای به عددی
    X = pd.get_dummies(X, drop_first=True)

    # ساختن متغیرهای تعاملی
    interaction_terms = []
    columns = list(X.columns)

    for var1, var2 in combinations(columns, 2):
        interaction_name = f'{var1}_&_{var2}'
        interaction_terms.append(X[var1] * X[var2])
        interaction_terms[-1].name = interaction_name

    # اضافه کردن متغیرهای تعاملی
    X = pd.concat([X] + interaction_terms, axis=1)

    # افزودن ستون ثابت
    X = sm.add_constant(X)

    # برازش مدل
    model = sm.OLS(y, X).fit()

    return model

In [ ]:
df = merged2.copy()
#df.head()
model = regression_with_interactions(df, ["Income"], "Net_Expenditure")

In [ ]:
from sklearn.linear_model import LinearRegression


def exp_regression(df, independent_vars, target_var, include_income=False):
    # انتخاب داده‌ها
    X = df[independent_vars]
    y = df[target_var]

    # تبدیل متغیرهای رشته‌ای به عددی
    X = pd.get_dummies(X, drop_first=True)
    
    if include_income:
        columns = list(X.columns)
        X['income'] = df["Income"]

        # ساخت متغیرهای تعاملی
        interaction_terms = []

        for var1 in columns:
            interaction_name = f'{var1}:Income'
            interaction_terms.append(X[var1] * X['income'])
            interaction_terms[-1].name = interaction_name

        # اضافه کردن متغیرهای تعاملی به X
        X = pd.concat([X] + interaction_terms, axis=1)

    # اضافه کردن یک ستون ثابت (intercept) برای مدل
    X['constant'] = 1

    # ساخت مدل رگرسیونی
    model = LinearRegression()

    # برازش مدل
    model.fit(X, y)
    
    print(model.score(X, y))
    print(model.predict(X))
    print(y)
    print()
    

    # نمایش ضرایب
    coef_df = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
    return coef_df

In [ ]:
wo_income = exp_regression(df, ["Education_Level"], "Net_Expenditure")
with_income = exp_regression(df, ["Education_Level"], "Net_Expenditure", True)

In [ ]:
merged_regr = pd.merge(wo_income, with_income, 'outer', left_index=True, right_index=True).sort_index()
display(merged_regr)

In [ ]:
merged_regr.plot(kind="bar", figsize=(12, 6))
plt.show()

In [ ]:
exp_in = merged2[["Net_Expenditure", "Income"]]
orig_size = exp_in.count()
exp_in = exp_in[exp_in["Net_Expenditure"] <= 1e8]
exp_in = exp_in[exp_in["Income"] >= 0]
exp_in = exp_in[exp_in["Income"] <= 1e9]
filtered_size = exp_in.count()
print(filtered_size/orig_size)

import matplotlib.pyplot as plt

exp_in.plot.scatter(x="Income", y="Net_Expenditure")

sm_model = sm.OLS(exp_in["Net_Expenditure"], exp_in["Income"]).fit()

m, b = np.polyfit(exp_in["Income"], exp_in["Net_Expenditure"], deg=1)
plt.axline(xy1=(0, b), slope=m, label=f'$y = {m:.1f}x {b:+.1f}$')

plt.xlabel("income")
plt.ylabel("entertainment expenditures")

plt.tight_layout()
plt.show()

In [ ]:
merged2["Income"].median()

In [ ]:
sm_model.summary()

In [ ]:
def regression(df, independent_vars, target_var, include_income=False):
    # انتخاب داده‌ها
    X = df[independent_vars]
    y = df[target_var]

    # تبدیل متغیرهای رشته‌ای به عددی
    X = pd.get_dummies(X, drop_first=True, dtype=int)

    if include_income:
        columns = list(X.columns)
        X['Income'] = df["Income"]
    
        # ساختن متغیرهای تعاملی
        interaction_terms = []

        for var1 in columns:
            interaction_name = f'{var1}_&_Income'
            interaction_terms.append(X[var1] * X['Income'])
            interaction_terms[-1].name = interaction_name

        # اضافه کردن متغیرهای تعاملی
        X = pd.concat([X] + interaction_terms, axis=1)

    
    # افزودن ستون ثابت
    X = sm.add_constant(X)
    # برازش مدل
    model = sm.OLS(y, X).fit()

    return model

In [ ]:
df2 = df.copy()
df2 = df2[df2["Education_Level"] != "Unofficial"]
model = regression(df2, ["Education_Level"], "Net_Expenditure", False)

In [ ]:
model.summary()

In [ ]:
from utils import *

def do_entertainment_for_year(year):
    expenditures_yy = csvfind(tableName="Expenditures", year=year)
    members_properties_yy = csvfind(tableName="members_properties", year=year)
    expenditures_yy_merged = expenditures_yy \
        [expenditures_yy["Table_Name"].isin(("entertainment",))] \
        .groupby(["ID"]) \
        ["Net_Expenditure"] \
        .sum()

    merged = pd.merge(get_total_income(year), expenditures_yy_merged, on="ID")

    head_edu = members_properties_yy[members_properties_yy["Relationship"] == "Head"][["ID", "Education_Level"]]

    merged = pd.merge(merged, head_edu, on=["ID"])

    df = merged.copy()
    #df = df[df["Education_Level"] != "Unofficial"]
    model_wo_income = regression(df, ["Education_Level"], "Net_Expenditure", False)
    model_with_income = regression(df, ["Education_Level"], "Net_Expenditure", True)
    display(model_wo_income.summary())
    display(model_with_income.summary())
    return (model_wo_income, model_with_income)

In [ ]:
do_entertainment_for_year(1395)

In [ ]:
do_entertainment_for_year(1402)

In [ ]:
from utils import *

def plot_for_year(year, xcut, ycut):
    expenditures_yy = csvfind(tableName="Expenditures", year=year)
    members_properties_yy = csvfind(tableName="members_properties", year=year)
    expenditures_yy_merged = expenditures_yy \
        [expenditures_yy["Table_Name"].isin(("entertainment",))] \
        .groupby(["ID"]) \
        ["Net_Expenditure"] \
        .sum()

    merged = pd.merge(get_total_income(year), expenditures_yy_merged, on="ID")

    head_edu = members_properties_yy[members_properties_yy["Relationship"] == "Head"][["ID", "Education_Level"]]

    merged = pd.merge(merged, head_edu, on=["ID"])

    df = merged.copy()

    exp_in = df[["Net_Expenditure", "Income"]]
    
    print(exp_in.mean())
    
    orig_size = exp_in.count()
    exp_in = exp_in[exp_in["Net_Expenditure"] <= ycut]
    exp_in = exp_in[exp_in["Income"] >= 0]
    exp_in = exp_in[exp_in["Income"] <= xcut]
    filtered_size = exp_in.count()
    print(filtered_size/orig_size)

    import matplotlib.pyplot as plt

    exp_in.plot.scatter(x="Income", y="Net_Expenditure")

    sm_model = sm.OLS(exp_in["Net_Expenditure"], exp_in["Income"]).fit()

    m, b = np.polyfit(exp_in["Income"], exp_in["Net_Expenditure"], deg=1)
    plt.axline(xy1=(0, b), slope=m, label=f'$y = {m:.1f}x {b:+.1f}$')

    plt.xlabel("income")
    plt.ylabel("entertainment expenditures")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_for_year(1392, 8e8, 3e7)

In [ ]:
plot_for_year(1402, 1e10, 3e8)

In [ ]:
import seaborn as sns

def get_overall_plot():
    dataframes = []
    for year in range(1392, 1403):
        expenditures_yy = csvfind(tableName="Expenditures", year=year)
        members_properties_yy = csvfind(tableName="members_properties", year=year)
        expenditures_yy_merged = expenditures_yy \
            [expenditures_yy["Table_Name"].isin(("entertainment",))] \
            .groupby(["ID"]) \
            ["Net_Expenditure"] \
            .sum()

        merged = pd.merge(get_total_income(year), expenditures_yy_merged, on="ID")

        df = merged.copy()

        df = df[["Net_Expenditure", "Income"]]
        df.rename(columns={"Income": "I", "Net_Expenditure": "E"}, inplace=True)
        df = df.assign(Year=year)
        dataframes.append(df)
    cdf = pd.concat(dataframes)    
    mdf = pd.melt(cdf, id_vars=['Year'], var_name='Class')
    plt.figure()
    ax = sns.boxplot(x="Year", y="value", hue="Class", data=mdf, showfliers=False, log_scale=True)
    plt.legend(loc="upper right")
    plt.show()

In [ ]:
get_overall_plot()

In [ ]:
csvfind(tableName="members_properties", year=1400).head()

In [ ]:
hbsir.load_table("members_properties", [1395])